# Refinement Model — Phase 3: Total Variation Regularization

Fine-tune from Phase 2 checkpoint with TV loss added to fight VOI degradation.

**Problem:** Phase 2 model improves TopoScore (+0.015) and SurfaceDice (+0.033) but
badly degrades VOI (-0.131). VOI measures segmentation compactness — the model produces
fragmented outputs because nothing in the Phase 2 loss penalizes spatial discontinuity.

**Solution:** Add Total Variation (TV) loss as regularization. TV penalizes differences
between adjacent voxels, directly fighting the fragmentation that kills VOI.

**Loss:** Phase2Loss + 0.1 * TV (keep Phase 2 balance, add gentle smoothness)

**Training:** Fine-tune from Phase 2 weights, lower LR, fit_flat_cos, 30 epochs

## 1. Imports & Config

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from fastai.learner import Learner
from fastai.callback.schedule import fit_flat_cos
from fastai.callback.tracker import SaveModelCallback
from fastai.callback.fp16 import MixedPrecision
from fastai.data.core import DataLoaders
from fastai.metrics import Metric
from scipy.ndimage import distance_transform_edt, label as scipy_label
from scipy.ndimage import binary_closing, generate_binary_structure, binary_propagation
from scipy.ndimage import zoom as scipy_zoom
from topometrics.leaderboard import compute_leaderboard_score
import random
import time
import copy

# Config
ROOT = Path("/workspace/vesuvius-kaggle-competition")
TRAIN_LBL = ROOT / "data" / "train_labels"
PROBMAP_DIR = ROOT / "data" / "refinement_data" / "probmaps"
CKPT_DIR = ROOT / "checkpoints"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOL_SIZE = 320
PATCH_SIZE = 160
BATCH_SIZE = 2
NUM_WORKERS = 4
PIN_MEMORY = True
SEED = 42

# Phase 3 hyperparameters
PHASE3_EPOCHS = 30
PHASE3_LR = 1e-4          # 3x lower than Phase 2 (fine-tuning)
TV_WEIGHT = 0.1            # TV regularization weight (gentle)
N_VAL_VOLUMES = 5
METRIC_DOWNSAMPLE = 4

# Baseline post-processing params
BASELINE_T_LOW = 0.35
BASELINE_T_HIGH = 0.75
CLOSING_Z_RADIUS = 2
CLOSING_XY_RADIUS = 1
DUST_MIN_SIZE = 100

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Phase 3: {PHASE3_EPOCHS} epochs, LR={PHASE3_LR}, TV_WEIGHT={TV_WEIGHT}")
print(f"Training: {PATCH_SIZE}^3 patches, bs={BATCH_SIZE}")

## 2. Model

In [ ]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class RefinementUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, channels=(8, 16, 32, 64), dropout=0.2):
        super().__init__()
        self.enc1 = ConvBlock3D(in_ch, channels[0])
        self.enc2 = ConvBlock3D(channels[0], channels[1])
        self.enc3 = ConvBlock3D(channels[1], channels[2])
        self.bottleneck = ConvBlock3D(channels[2], channels[3])
        self.dropout = nn.Dropout3d(p=dropout)
        self.dec3 = ConvBlock3D(channels[3] + channels[2], channels[2])
        self.dec2 = ConvBlock3D(channels[2] + channels[1], channels[1])
        self.dec1 = ConvBlock3D(channels[1] + channels[0], channels[0])
        self.conv_final = nn.Conv3d(channels[0], out_ch, 1)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        b = self.dropout(b)
        d3 = F.interpolate(b, size=e3.shape[2:], mode='trilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = F.interpolate(d3, size=e2.shape[2:], mode='trilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = F.interpolate(d2, size=e1.shape[2:], mode='trilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.conv_final(d1)


# Load Phase 2 checkpoint
model = RefinementUNet3D().to(DEVICE)
phase2_path = CKPT_DIR / "models" / "best_refinement_phase2.pth"
state_dict = torch.load(phase2_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict)
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Phase 2 checkpoint: {phase2_path}")
print(f"RefinementUNet3D: {n_params:,} params")

## 3. Dataset

In [ ]:
def compute_signed_distance_map(label, mask):
    fg = label.astype(bool)
    bg = ~fg
    dist_inside = distance_transform_edt(fg)
    dist_outside = distance_transform_edt(bg)
    signed_dist = dist_outside - dist_inside
    max_dist = max(np.abs(signed_dist).max(), 1.0)
    signed_dist = signed_dist / max_dist
    signed_dist = signed_dist * mask
    return signed_dist.astype(np.float32)


class RefinementDataset(Dataset):
    def __init__(self, ids, probmap_dir, lbl_dir, patch_size=160,
                 augment=False, compute_dist=False):
        self.ids = ids
        self.probmap_dir = Path(probmap_dir)
        self.lbl_dir = Path(lbl_dir)
        self.patch_size = patch_size
        self.augment = augment
        self.compute_dist = compute_dist

    def __len__(self):
        return len(self.ids)

    def _random_crop(self, *arrays):
        ps = self.patch_size
        d, h, w = arrays[0].shape
        z = random.randint(0, d - ps)
        y = random.randint(0, h - ps)
        x = random.randint(0, w - ps)
        return [a[z:z+ps, y:y+ps, x:x+ps] for a in arrays]

    def __getitem__(self, idx):
        vol_id = self.ids[idx]
        prob = np.load(self.probmap_dir / f"{vol_id}.npy")
        lbl = tifffile.imread(self.lbl_dir / f"{vol_id}.tif")

        if self.compute_dist:
            mask_f32 = (lbl != 2).astype(np.float32)
            label_f32 = (lbl == 1).astype(np.float32)
            dist_map = compute_signed_distance_map(label_f32, mask_f32).astype(np.float16)
            prob, lbl, dist_map = self._random_crop(prob, lbl, dist_map)
        else:
            prob, lbl = self._random_crop(prob, lbl)
            dist_map = None

        if self.augment:
            for axis in range(3):
                if random.random() > 0.5:
                    prob = np.flip(prob, axis=axis).copy()
                    lbl = np.flip(lbl, axis=axis).copy()
                    if dist_map is not None:
                        dist_map = np.flip(dist_map, axis=axis).copy()
            k = random.randint(0, 3)
            plane_axes = random.choice([(0, 1), (0, 2), (1, 2)])
            if k > 0:
                prob = np.rot90(prob, k=k, axes=plane_axes).copy()
                lbl = np.rot90(lbl, k=k, axes=plane_axes).copy()
                if dist_map is not None:
                    dist_map = np.rot90(dist_map, k=k, axes=plane_axes).copy()

        prob = torch.from_numpy(prob.copy()).unsqueeze(0)
        lbl_t = torch.from_numpy(lbl.copy()).unsqueeze(0)
        if dist_map is not None:
            dist_t = torch.from_numpy(dist_map.copy()).unsqueeze(0)
        else:
            dist_t = torch.zeros(1, dtype=torch.float16)

        return prob, (lbl_t, dist_t)

## 4. DataLoaders

In [ ]:
train_df = pd.read_csv(ROOT / "data" / "train.csv")
available_ids = set(int(p.stem) for p in PROBMAP_DIR.glob("*.npy")) & \
                set(int(p.stem) for p in TRAIN_LBL.glob("*.tif"))
train_df = train_df[train_df.id.isin(available_ids)].reset_index(drop=True)

VAL_SCROLL = 26002
val_ids = train_df[train_df.scroll_id == VAL_SCROLL].id.tolist()
trn_ids = train_df[train_df.scroll_id != VAL_SCROLL].id.tolist()

print(f"Train: {len(trn_ids)}, Val: {len(val_ids)}")

# Phase 3 needs distance maps (for Boundary loss component)
trn_ds = RefinementDataset(trn_ids, PROBMAP_DIR, TRAIN_LBL,
                            patch_size=PATCH_SIZE, augment=True, compute_dist=True)
val_ds = RefinementDataset(val_ids, PROBMAP_DIR, TRAIN_LBL,
                            patch_size=PATCH_SIZE, augment=False, compute_dist=True)

trn_dl = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

dls = DataLoaders(trn_dl, val_dl, device=DEVICE)
print(f"DataLoaders ready (Phase 3: {PATCH_SIZE}^3 patches, compute_dist=True)")

## 5. Phase 3 Loss: Phase 2 + Total Variation

In [ ]:
# --- Soft skeleton for clDice ---

def soft_erode_3d(img):
    return -F.max_pool3d(-img, kernel_size=3, stride=1, padding=1)

def soft_dilate_3d(img):
    return F.max_pool3d(img, kernel_size=3, stride=1, padding=1)

def soft_open_3d(img):
    return soft_dilate_3d(soft_erode_3d(img))

def soft_skeleton_3d(img, iters=5):
    skel = F.relu(img - soft_open_3d(img))
    for _ in range(iters):
        img = soft_erode_3d(img)
        delta = F.relu(img - soft_open_3d(img))
        skel = skel + delta
    return skel

def soft_cldice(pred_sig, target, iters=5, smooth=1.0):
    pred_ds = F.avg_pool3d(pred_sig, kernel_size=2)
    with torch.no_grad():
        target_ds = F.avg_pool3d(target, kernel_size=2)
    skel_pred = soft_skeleton_3d(pred_ds, iters=iters)
    with torch.no_grad():
        skel_target = soft_skeleton_3d(target_ds, iters=iters)
    tprec = ((skel_pred * target_ds).sum() + smooth) / (skel_pred.sum() + smooth)
    tsens = ((skel_target * pred_ds).sum() + smooth) / (skel_target.sum() + smooth)
    return 2.0 * tprec * tsens / (tprec + tsens + 1e-8)


def boundary_loss(pred_sig, dist_map, mask):
    return (pred_sig * dist_map * mask).sum() / (mask.sum() + 1e-8)


def unpack_targets(target_tuple):
    lbl_u8, dist_data = target_tuple
    label = (lbl_u8 == 1).float()
    mask = (lbl_u8 != 2).float()
    dist_map = dist_data.float()
    return label, mask, dist_map


def total_variation_3d(pred_sig):
    """Total Variation: sum of absolute differences between adjacent voxels.
    Penalizes spatial discontinuity (fragmentation) in predictions."""
    tv_z = torch.abs(pred_sig[:, :, 1:, :, :] - pred_sig[:, :, :-1, :, :]).mean()
    tv_y = torch.abs(pred_sig[:, :, :, 1:, :] - pred_sig[:, :, :, :-1, :]).mean()
    tv_x = torch.abs(pred_sig[:, :, :, :, 1:] - pred_sig[:, :, :, :, :-1]).mean()
    return tv_z + tv_y + tv_x


class Phase3Loss(nn.Module):
    """Phase 2 loss + Total Variation regularization.
    
    Phase 2: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary
    Phase 3: Phase2 + tv_weight * TV
    
    TV is added on top (not replacing any component) so the existing
    balance that produces good topo+sdice is preserved.
    """
    def __init__(self, tv_weight=0.1, smooth=1.0, skel_iters=5):
        super().__init__()
        self.tv_weight = tv_weight
        self.smooth = smooth
        self.skel_iters = skel_iters

    def forward(self, pred, target_tuple):
        target, mask, dist_map = unpack_targets(target_tuple)
        pred_sig = torch.sigmoid(pred)
        pred_masked = pred_sig * mask
        target_masked = target * mask

        # BCE (masked)
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        bce = (bce * mask).sum() / (mask.sum() + 1e-8)

        # Dice (masked)
        intersection = (pred_masked * target_masked).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_masked.sum() + target_masked.sum() + self.smooth
        )
        dice_loss = 1.0 - dice

        # clDice (masked)
        cldice_loss = 1.0 - soft_cldice(pred_masked, target_masked,
                                         iters=self.skel_iters, smooth=self.smooth)

        # Boundary loss
        bd_loss = boundary_loss(pred_sig, dist_map, mask)

        # Total Variation (on sigmoid output, full volume including unlabeled)
        tv_loss = total_variation_3d(pred_sig)

        # Phase 2 components (unchanged) + TV regularization
        phase2 = 0.2 * bce + 0.2 * dice_loss + 0.3 * cldice_loss + 0.3 * bd_loss
        total = phase2 + self.tv_weight * tv_loss

        return total


# Smoke test
loss_fn = Phase3Loss(tv_weight=TV_WEIGHT)
with torch.no_grad():
    dummy_pred = torch.randn(1, 1, 32, 32, 32).cuda()
    dummy_lbl = torch.randint(0, 3, (1, 1, 32, 32, 32), dtype=torch.uint8).cuda()
    dummy_dist = torch.randn(1, 1, 32, 32, 32, dtype=torch.float16).cuda()
    loss_val = loss_fn(dummy_pred, (dummy_lbl, dummy_dist))
    print(f"Phase 3 loss smoke test: {loss_val.item():.4f}")
    
    # Also test TV alone to see its magnitude
    tv_val = total_variation_3d(torch.sigmoid(dummy_pred))
    print(f"TV component alone: {tv_val.item():.4f}")
    print(f"TV contribution to loss: {TV_WEIGHT * tv_val.item():.4f}")

print(f"\nPhase 3 loss: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary + {TV_WEIGHT}*TV")

## 6. Metrics

In [ ]:
class MaskedDiceMetric(Metric):
    def __init__(self):
        self.reset()

    def reset(self):
        self.intersection = 0.0
        self.union = 0.0

    def accumulate(self, learn):
        pred = torch.sigmoid(learn.pred) > 0.5
        lbl_u8, _dist = learn.y
        target = lbl_u8 == 1
        mask = lbl_u8 != 2
        pred_m = pred & mask
        tgt_m = target & mask
        self.intersection += (pred_m & tgt_m).sum().item()
        self.union += pred_m.sum().item() + tgt_m.sum().item()

    @property
    def value(self):
        return (2.0 * self.intersection + 1.0) / (self.union + 1.0)

    @property
    def name(self):
        return "dice"


class RefinementCompetitionMetric(Metric):
    """Competition metric: full 320^3 inference on CPU, comp_score on val volumes."""
    def __init__(self, val_ids, probmap_dir, lbl_dir, n_volumes=5, downsample=4):
        self.val_ids = val_ids[:n_volumes]
        self.probmap_dir = Path(probmap_dir)
        self.lbl_dir = Path(lbl_dir)
        self.downsample = downsample
        self._model = None
        self.reset()

    def reset(self):
        self._model = None

    def accumulate(self, learn):
        self._model = learn.model

    @property
    def value(self):
        if self._model is None:
            return 0.0

        cpu_model = copy.deepcopy(self._model).cpu().float().eval()
        scores = []
        for vid in self.val_ids:
            prob = np.load(self.probmap_dir / f"{vid}.npy").astype(np.float32)
            lbl = tifffile.imread(self.lbl_dir / f"{vid}.tif")

            with torch.no_grad():
                inp = torch.from_numpy(prob).unsqueeze(0).unsqueeze(0)
                logits = cpu_model(inp)
                pred = (torch.sigmoid(logits).squeeze().numpy() > 0.5).astype(np.uint8)

            ds = self.downsample
            if ds > 1:
                scale = 1.0 / ds
                pred = scipy_zoom(pred, scale, order=0).astype(np.uint8)
                lbl = scipy_zoom(lbl, scale, order=0).astype(np.uint8)

            report = compute_leaderboard_score(
                pred, lbl, ignore_label=2, spacing=(1, 1, 1),
                surface_tolerance=2.0, voi_alpha=0.3,
                combine_weights=(0.3, 0.35, 0.35),
            )
            scores.append(report.score)

        del cpu_model
        return float(np.mean(scores))

    @property
    def name(self):
        return "comp_score"


print("Metrics defined: dice, comp_score")

## 7. Phase 3 Training

In [ ]:
loss_fn = Phase3Loss(tv_weight=TV_WEIGHT)

comp_metric = RefinementCompetitionMetric(
    val_ids, PROBMAP_DIR, TRAIN_LBL,
    n_volumes=N_VAL_VOLUMES, downsample=METRIC_DOWNSAMPLE,
)

learn = Learner(
    dls,
    model,
    loss_func=loss_fn,
    metrics=[MaskedDiceMetric(), comp_metric],
    cbs=[
        MixedPrecision(),
        SaveModelCallback(monitor="comp_score", comp=np.greater,
                          fname="best_refinement_phase3"),
    ],
    path=CKPT_DIR,
)

print(f"Phase 3 Learner ready.")
print(f"Loss: Phase2 + {TV_WEIGHT}*TV")
print(f"Starting from Phase 2 weights, LR={PHASE3_LR}, fit_flat_cos")
print(f"Save by comp_score (higher is better)")

In [ ]:
print(f"Training Phase 3: {PHASE3_EPOCHS} epochs, LR={PHASE3_LR}, fit_flat_cos")
learn.fit_flat_cos(PHASE3_EPOCHS, lr=PHASE3_LR)

In [ ]:
learn.recorder.plot_loss()
plt.title("Phase 3: Phase2 + TV Regularization")
plt.show()

## 8. Done

Training complete. Evaluation and export run separately via overnight script:
```bash
python scripts/eval_refinement.py --phase 3 --n-volumes 20
```